In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
data_dir = Path("../data/bin")
names = ["actual_et", "actual_eto", "actual_etof", "ndvi"]
dt = pd.DataFrame(columns=['field_id', 'time'])

tables = [pd.DataFrame(columns=['field_id', 'time', name]) for name in names]
for item in range(0, len(names)):
    name = names[item]
    # Collect list of files whose name contains the current column name
    files = (data_dir / "20250829_220115").glob(f'*.{name}.csv')

    # Iterate through each file through Generator iterator
    for file in files:
        # e.g. CA_270812.27.actual_eto.csv
        # becomes ['CA_270812', '27', 'actual_eto', 'csv']
        parts = str(file.name).split('.')
        # Contains [time, {variable}]
        data = pd.read_csv(file, header=0, names=['time', name])
        data['field_id'] = parts[0]
        tables[item] = pd.concat([data, tables[item]], ignore_index=True)

for table in tables:
    # Conducts full outer joins to preserve time column not always overlapping.
    dt = dt.merge(table, on=['field_id', 'time'], how='outer')

dt

/tmp/ipykernel_1214853/426125839.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  tables[item] = pd.concat([data, tables[item]], ignore_index=True)
/tmp/ipykernel_1214853/426125839.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  tables[item] = pd.concat([data, tables[item]], ignore_index=True)
/tmp/ipykernel_1214853/426125839.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or

,field_id,time,actual_et,actual_eto,actual_etof,ndvi
0,CA_244000,2016-01-01,0.556,1.149,0.484,0.162
1,CA_244000,2016-01-02,0.715,1.461,0.489,0.165
2,CA_244000,2016-01-03,0.809,1.635,0.495,0.168
3,CA_244000,2016-01-04,0.582,1.164,0.500,0.171
4,CA_244000,2016-01-05,0.634,1.255,0.505,0.173
...,...,...,...,...,...,...
6371195,CA_42345,2025-08-16,5.444,5.909,0.921,0.761
6371196,CA_42345,2025-08-17,5.301,5.754,0.921,0.761
6371197,CA_42345,2025-08-18,5.844,6.343,0.921,0.761
6371198,CA_42345,2025-08-19,6.973,7.569,0.921,0.761


In [3]:
dt.to_parquet(data_dir.parent / "ca_historical.parquet", index=False)

In [4]:
dt["time"] = pd.to_datetime(dt["time"])
# Create a column for day of year
dt["doy"] = dt["time"].dt.dayofyear
# Group by field, crop, and doy then calculate the average conditions
climatology_table = dt.groupby(["field_id", "doy"])[
    ["actual_et", "actual_eto", "actual_etof"]
].agg("mean")

climatology_table.reset_index().to_parquet(data_dir.parent / "ca_climatology.parquet", index=False)
climatology_table.reset_index()

,field_id,doy,actual_et,actual_eto,actual_etof
0,CA_244000,1,0.603900,0.942200,0.623600
1,CA_244000,2,0.644300,1.061200,0.621600
2,CA_244000,3,0.846500,1.316800,0.619500
3,CA_244000,4,0.824300,1.288200,0.617300
4,CA_244000,5,0.722400,1.202700,0.615200
...,...,...,...,...,...
662455,CA_42345,362,0.467333,1.092444,0.423889
662456,CA_42345,363,0.428111,1.023778,0.422222
662457,CA_42345,364,0.440778,1.036222,0.420333
662458,CA_42345,365,0.346556,0.820222,0.419778


In [5]:
avgs_table = (
dt.loc[(dt["time"].dt.year >= 2024), :]
    .groupby(["field_id"])[["actual_et", "actual_eto", "actual_etof"]]
    .agg("mean")
)
avgs_table.reset_index().to_parquet(data_dir.parent / "ca_avgs.parquet", index=False)
avgs_table.reset_index()

,field_id,actual_et,actual_eto,actual_etof
0,CA_244000,2.558599,4.723047,0.568691
1,CA_244018,1.367759,4.523244,0.425052
2,CA_244025,2.132244,4.584475,0.464321
3,CA_244035,2.568796,4.667490,0.525247
4,CA_244053,1.985793,4.565493,0.528950
...,...,...,...,...
1805,CA_42273,4.090510,4.430236,0.926202
1806,CA_42274,2.920385,4.434871,0.735709
1807,CA_42289,2.646022,4.440930,0.630657
1808,CA_42338,3.214522,4.433997,0.684114
